# BioT5 Mini Dataset Review On Kaggle

This notebook clones the repo, ensures ChEBI-20 exists locally, samples 128 random train descriptions for the native BioT5 review flow, and also packages a small grouped multi-molecule dataset for post-training debugging.
It keeps the same review-format outputs used by the local review notebooks while exporting a second `mini-post-training.zip` artifact that contains ready-to-train grouped split files.


## Notes

- Enable internet so Kaggle can clone the repo, fetch the BioT5 checkpoint, and load the diverse beam backend if needed.
- Enable a GPU accelerator because BioT5 generation is expensive on CPU.
- Add an optional Kaggle secret named `HF_TOKEN` if you want authenticated Hugging Face access.
- The review outputs are written to `outputs/kaggle/mini_dataset/`, exported under `/kaggle/working/thesis_artifacts/mini_dataset_biot5_review/`, and zipped for easier download.
- The post-training debug outputs are written to `outputs/kaggle/mini_post_training/`, exported under `/kaggle/working/thesis_artifacts/mini_post_training/`, and zipped as `/kaggle/working/thesis_artifacts/mini-post-training.zip`.
- To build the post-training debug zip, attach the merged grouped collection artifact dataset from `05_merge_biot5_collection_parts.ipynb` or keep `data/post_training/processed/train_multimol.jsonl` available in the repo clone.


In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/kaggle/working/Thesis")
REVIEW_STAGE_NAME = "mini_dataset_biot5_review"
MINI_POST_TRAINING_STAGE_NAME = "mini_post_training"
UPSTREAM_GROUPED_STAGE = "merge_biot5_collection_parts"

%cd /kaggle/working
!if [ -d "{REPO_DIR / '.git'}" ]; then echo "Reusing {REPO_DIR}"; elif [ -d "{REPO_DIR}" ]; then echo "Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook." && false; else git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" fetch --depth 1 origin "{REPO_BRANCH}" && git -C "{REPO_DIR}" checkout -B "{REPO_BRANCH}" FETCH_HEAD

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from thesis_kaggle_support import (
    ARTIFACT_EXPORT_ROOT,
    copy_stage_artifact_to_local,
    create_zip_archive,
    ensure_grouped_split_files,
    ensure_runtime_dependencies,
    export_stage_artifacts,
    import_or_none,
    json_dumps,
    pip_install,
    report_runtime,
    write_jsonl,
)


In [ ]:
CHEBI_OUTPUT_DIR = REPO_DIR / "data" / "chebi20"
CHEBI_PROCESSED_DIR = CHEBI_OUTPUT_DIR / "processed"
LOCAL_GROUPED_TRAIN_FILE = REPO_DIR / "data" / "post_training" / "processed" / "train_multimol.jsonl"

OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / "mini_dataset"
RAW_GENERATIONS_PATH = OUTPUT_DIR / "raw_generations.jsonl"
REVIEW_ROWS_PATH = OUTPUT_DIR / "review_rows.jsonl"
SUMMARY_ROWS_PATH = OUTPUT_DIR / "summary_rows.json"
REVIEW_CONFIG_SNAPSHOT_PATH = OUTPUT_DIR / "config_snapshot.json"

MINI_POST_TRAINING_OUTPUT_DIR = REPO_DIR / "outputs" / "kaggle" / "mini_post_training"
MINI_GROUPED_SAMPLE_PATH = MINI_POST_TRAINING_OUTPUT_DIR / "mini_grouped_train_multimol.jsonl"
MINI_POST_TRAINING_CONFIG_SNAPSHOT_PATH = MINI_POST_TRAINING_OUTPUT_DIR / "config_snapshot.json"
MINI_GROUPED_SPLIT_DIR = REPO_DIR / "kaggle" / "generated_data" / "mini_post_training"

REVIEW_ARTIFACT_ZIP_PATH = ARTIFACT_EXPORT_ROOT / f"{REVIEW_STAGE_NAME}.zip"
MINI_POST_TRAINING_ZIP_PATH = ARTIFACT_EXPORT_ROOT / "mini-post-training.zip"

ensure_runtime_dependencies(REPO_DIR)
if import_or_none("huggingface_hub") is None:
    pip_install("huggingface_hub")
if import_or_none("sentencepiece") is None:
    pip_install("sentencepiece")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_login_status = "not_attempted"
hf_token = None
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    hf_login_status = f"secret_unavailable:{exc.__class__.__name__}"

if hf_token:
    login(token=hf_token)
    hf_login_status = "logged_in"
elif hf_login_status == "not_attempted":
    hf_login_status = "no_hf_token"

runtime_report = report_runtime(require_gpu=True)

print(json_dumps({
    "runtime": runtime_report,
    "hf_login_status": hf_login_status,
    "repo_dir": str(REPO_DIR),
    "chebi_processed_dir": str(CHEBI_PROCESSED_DIR),
    "local_grouped_train_file": str(LOCAL_GROUPED_TRAIN_FILE),
    "review_output_dir": str(OUTPUT_DIR),
    "mini_post_training_output_dir": str(MINI_POST_TRAINING_OUTPUT_DIR),
    "review_artifact_zip_path": str(REVIEW_ARTIFACT_ZIP_PATH),
    "mini_post_training_zip_path": str(MINI_POST_TRAINING_ZIP_PATH),
}))


In [ ]:
have_processed = all((CHEBI_PROCESSED_DIR / f"{split}.jsonl").exists() for split in ("train", "validation", "test"))
print(json_dumps({
    "have_processed": have_processed,
    "processed_dir": str(CHEBI_PROCESSED_DIR),
}))

%cd {REPO_DIR}
!if [ -f "{CHEBI_PROCESSED_DIR / 'train.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'validation.jsonl'}" ] && [ -f "{CHEBI_PROCESSED_DIR / 'test.jsonl'}" ]; then echo "ChEBI processed splits already exist"; else python scripts/download_chebi20.py --output-dir "{CHEBI_OUTPUT_DIR}"; fi


In [ ]:
from __future__ import annotations

import json
import random
import shutil
import time

import selfies
import torch
import transformers

from data_collection import BioT5DiverseBeamGenerator
from molecules.collection.filtering import CollectionMetricConfig, prepare_reference_groups
from notebooks.biot5_collection_review_support import (
    assess_biot5_native_generation_output,
    summarize_biot5_native_review_records,
)
from src.io_utils import read_jsonl
from src.prompting import build_text2mol_prompt


def print_section(title: str) -> None:
    print(f"\n=== {title} ===")


def print_json(title: str, payload) -> None:
    print_section(title)
    print(json.dumps(payload, indent=2, ensure_ascii=False))


def print_records(title: str, records, *, limit: int | None = None, keys: list[str] | None = None) -> None:
    payload = list(records)
    total = len(payload)
    if keys is not None:
        payload = [{key: row.get(key) for key in keys} for row in payload]
    shown = payload if limit is None else payload[:limit]
    print_section(f"{title} (showing {len(shown)} of {total})")
    print(json.dumps(shown, indent=2, ensure_ascii=False))


version_report = {
    "python": str(__import__("sys").version.split()[0]),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "selfies": selfies.__version__,
    "repo_dir": str(REPO_DIR),
}
print_json("Environment", version_report)


In [ ]:
TRAIN_FILE = CHEBI_PROCESSED_DIR / "train.jsonl"
MODEL_NAME_OR_PATH = "QizhiPei/biot5-plus-base-chebi20"
DEVICE = "auto"

SELECTED_DESCRIPTION_COUNT = 128
MINI_POST_TRAINING_EXAMPLE_COUNT = 128
SELECTION_SEED = 42
VALIDATION_FRACTION = 0.05
TEST_FRACTION = 0.05
NUM_SAMPLES = 100
MODEL_MAX_LENGTH = 512
ALLOW_FILTER_FALLBACK = True

GENERATION_CONFIGS = {
    "greedy_native": {
        "target_count": 1,
        "max_length": 512,
        "num_beams": 1,
        "num_return_sequences": 1,
    },
    "diverse_beam_fast": {
        "target_count": NUM_SAMPLES,
        "max_length": 512,
        "num_beams": 30,
        "num_return_sequences": 30,
        "num_beam_groups": 5,
        "diversity_penalty": 0.3,
        "early_stopping": True,
        "length_penalty": 1.0,
    },
    "diverse_beam_alt": {
        "target_count": NUM_SAMPLES,
        "max_length": 512,
        "num_beams": 30,
        "num_return_sequences": 30,
        "num_beam_groups": 6,
        "diversity_penalty": 0.5,
        "early_stopping": True,
        "length_penalty": 1.0,
    },
}
METRIC_CONFIG = CollectionMetricConfig(
    fingerprint_radius=2,
    fingerprint_num_bits=2048,
    acceptance_dice_threshold=0.7,
)

config_preview = {
    "train_file": str(TRAIN_FILE),
    "local_grouped_train_file": str(LOCAL_GROUPED_TRAIN_FILE),
    "upstream_grouped_stage": UPSTREAM_GROUPED_STAGE,
    "model_name_or_path": MODEL_NAME_OR_PATH,
    "device": DEVICE,
    "selected_description_count": SELECTED_DESCRIPTION_COUNT,
    "mini_post_training_example_count": MINI_POST_TRAINING_EXAMPLE_COUNT,
    "selection_seed": SELECTION_SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "test_fraction": TEST_FRACTION,
    "num_samples": NUM_SAMPLES,
    "model_max_length": MODEL_MAX_LENGTH,
    "allow_filter_fallback": ALLOW_FILTER_FALLBACK,
    "generation_configs": GENERATION_CONFIGS,
    "review_output_dir": str(OUTPUT_DIR),
    "mini_post_training_output_dir": str(MINI_POST_TRAINING_OUTPUT_DIR),
    "review_artifact_zip_path": str(REVIEW_ARTIFACT_ZIP_PATH),
    "mini_post_training_zip_path": str(MINI_POST_TRAINING_ZIP_PATH),
}
print_json("Notebook Config", config_preview)


In [ ]:
if not LOCAL_GROUPED_TRAIN_FILE.exists():
    copied_grouped_train = copy_stage_artifact_to_local(
        stage_name=UPSTREAM_GROUPED_STAGE,
        artifact_relpath="post_training_processed/train_multimol.jsonl",
        local_path=LOCAL_GROUPED_TRAIN_FILE,
    )
else:
    copied_grouped_train = None

if not LOCAL_GROUPED_TRAIN_FILE.exists():
    raise FileNotFoundError(
        f"Grouped training file not found at {LOCAL_GROUPED_TRAIN_FILE}. "
        "Attach the merged collection artifact dataset or run 05_merge_biot5_collection_parts.ipynb first."
    )

all_grouped_records = read_jsonl(LOCAL_GROUPED_TRAIN_FILE)
if len(all_grouped_records) < MINI_POST_TRAINING_EXAMPLE_COUNT:
    raise ValueError(
        f"Requested {MINI_POST_TRAINING_EXAMPLE_COUNT} grouped examples, but only {len(all_grouped_records)} are available."
    )

mini_grouped_rng = random.Random(SELECTION_SEED)
selected_grouped_records = mini_grouped_rng.sample(
    all_grouped_records,
    k=MINI_POST_TRAINING_EXAMPLE_COUNT,
)
SELECTED_GROUPED_EXAMPLE_IDS = [
    str(record.get("id") or "unknown")
    for record in selected_grouped_records
]

MINI_POST_TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(MINI_GROUPED_SAMPLE_PATH, selected_grouped_records)

if MINI_GROUPED_SPLIT_DIR.exists():
    shutil.rmtree(MINI_GROUPED_SPLIT_DIR)

mini_split_paths = ensure_grouped_split_files(
    MINI_GROUPED_SAMPLE_PATH,
    MINI_GROUPED_SPLIT_DIR,
    seed=SELECTION_SEED,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
)
mini_split_counts = {
    name: len(read_jsonl(path))
    for name, path in mini_split_paths.items()
}

mini_grouped_preview = [
    {
        "id": str(record.get("id") or "unknown"),
        "description": str(record.get("description") or ""),
        "num_targets": len(record.get("target_selfies_list", [])),
    }
    for record in selected_grouped_records
]
print_json(
    "Mini post-training dataset",
    {
        "copied_grouped_train": None if copied_grouped_train is None else str(copied_grouped_train),
        "grouped_train_file": str(LOCAL_GROUPED_TRAIN_FILE),
        "num_available_grouped_records": len(all_grouped_records),
        "mini_post_training_example_count": len(selected_grouped_records),
        "selection_seed": SELECTION_SEED,
        "selected_grouped_example_ids_preview": SELECTED_GROUPED_EXAMPLE_IDS[:20],
        "mini_grouped_sample_path": str(MINI_GROUPED_SAMPLE_PATH),
        "mini_split_paths": {name: str(path) for name, path in mini_split_paths.items()},
        "mini_split_counts": mini_split_counts,
    },
)
print_records("Mini post-training grouped preview", mini_grouped_preview, limit=20)


In [ ]:
all_train_records = read_jsonl(TRAIN_FILE)
records_by_id = {
    str(record.get("id")): record
    for record in all_train_records
    if record.get("id") is not None
}
available_description_ids = list(records_by_id)
if len(available_description_ids) < SELECTED_DESCRIPTION_COUNT:
    raise ValueError(
        f"Requested {SELECTED_DESCRIPTION_COUNT} description IDs, but only {len(available_description_ids)} are available."
    )

selection_rng = random.Random(SELECTION_SEED)
SELECTED_DESCRIPTION_IDS = selection_rng.sample(available_description_ids, k=SELECTED_DESCRIPTION_COUNT)
selected_records = [records_by_id[record_id] for record_id in SELECTED_DESCRIPTION_IDS]
print_json(
    "Selected ID Report",
    {
        "num_available_description_ids": len(available_description_ids),
        "selected_description_count": len(SELECTED_DESCRIPTION_IDS),
        "selection_seed": SELECTION_SEED,
        "selected_description_ids_preview": SELECTED_DESCRIPTION_IDS[:20],
    },
)

reference_groups = prepare_reference_groups(all_train_records, METRIC_CONFIG)

selected_preview = [
    {
        "id": str(record.get("id")),
        "description": str(record.get("description")),
        "reference_selfies": str(record.get("selfies")),
        "reference_smiles": str(record.get("source_smiles")),
    }
    for record in selected_records
]
print_records("Selected descriptions", selected_preview, limit=20)

prompt_variants_by_id = {}
for record in selected_records:
    description_id = str(record.get("id"))
    description = str(record.get("description"))
    prompt_variants_by_id[description_id] = {
        "native_selfies": build_text2mol_prompt(description),
    }

prompt_preview = []
for record in selected_records:
    description_id = str(record.get("id"))
    for prompt_variant, prompt_text in prompt_variants_by_id[description_id].items():
        prompt_preview.append(
            {
                "description_id": description_id,
                "prompt_variant": prompt_variant,
                "prompt_text": prompt_text,
            }
        )
print_records("Prompt preview", prompt_preview, limit=20)


In [ ]:
# Use the shared active collection generator so the review notebook matches collection-time inference.


In [ ]:
first_generation_config_name = next(iter(GENERATION_CONFIGS))
generator = BioT5DiverseBeamGenerator(
    model_name_or_path=MODEL_NAME_OR_PATH,
    device_name=DEVICE,
    model_max_length=MODEL_MAX_LENGTH,
    generation_config=GENERATION_CONFIGS[first_generation_config_name],
)

print_json("Generator Preview", {
    "device": str(generator.device),
    "supports_remote_group_beam_search": bool(generator.supports_remote_group_beam_search),
    "num_selected_descriptions": len(selected_records),
    "generation_config_names": list(GENERATION_CONFIGS.keys()),
})


In [ ]:
raw_generations = []
for generation_config_name, generation_config in GENERATION_CONFIGS.items():
    generator.generation_config = dict(generation_config)
    target_count = int(generation_config.get("target_count", NUM_SAMPLES))
    print_section(f"Running {generation_config_name}")

    for record in selected_records:
        description_id = str(record.get("id"))
        description = str(record.get("description"))
        for prompt_variant, prompt_text in prompt_variants_by_id[description_id].items():
            started_at = time.perf_counter()
            outputs = generator.generate_candidates(prompt_text, target_count)
            elapsed_seconds_batch = time.perf_counter() - started_at
            seconds_per_sample_batch = elapsed_seconds_batch / max(len(outputs), 1)
            print(
                f"{generation_config_name} | {description_id} | {prompt_variant} | "
                f"samples={len(outputs)} | elapsed_seconds={elapsed_seconds_batch:.2f} | "
                f"seconds_per_sample={seconds_per_sample_batch:.3f}"
            )
            for candidate_index, raw_prediction_text in enumerate(outputs):
                raw_generations.append(
                    {
                        "generation_config_name": generation_config_name,
                        "description_id": description_id,
                        "description": description,
                        "prompt_variant": prompt_variant,
                        "candidate_index": candidate_index,
                        "raw_prediction_text": raw_prediction_text,
                        "elapsed_seconds_batch": elapsed_seconds_batch,
                        "seconds_per_sample_batch": seconds_per_sample_batch,
                    }
                )

print_json("Generation Report", {
    "num_rows": len(raw_generations),
    "generation_configs": sorted({row["generation_config_name"] for row in raw_generations}),
    "description_ids": sorted({row["description_id"] for row in raw_generations}),
    "prompt_variants": sorted({row["prompt_variant"] for row in raw_generations}),
})
print_records(
    "Raw generation preview",
    raw_generations,
    limit=12,
    keys=[
        "generation_config_name",
        "description_id",
        "prompt_variant",
        "candidate_index",
        "raw_prediction_text",
    ],
)


In [ ]:
review_rows = []
for row in raw_generations:
    review_rows.append(
        assess_biot5_native_generation_output(
            description_id=row["description_id"],
            description=row["description"],
            prompt_variant=row["prompt_variant"],
            candidate_index=row["candidate_index"],
            raw_prediction_text=row["raw_prediction_text"],
            references=reference_groups[row["description"]],
            metric_config=METRIC_CONFIG,
            generation_config_name=row["generation_config_name"],
            elapsed_seconds_batch=row["elapsed_seconds_batch"],
            seconds_per_sample_batch=row["seconds_per_sample_batch"],
            allow_filter_fallback=ALLOW_FILTER_FALLBACK,
        )
    )

parsed_preview_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "candidate_index",
    "raw_prediction_text",
    "cleaned_selfies",
    "parsed_selfies",
    "filtered_selfies",
    "used_filter_selfies_fallback",
    "decoded_smiles",
    "selfies_decode_error",
]
print_records("Raw vs Parsed SELFIES preview", review_rows, limit=20, keys=parsed_preview_columns)

detail_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "candidate_index",
    "raw_prediction_text",
    "cleaned_selfies",
    "parsed_selfies",
    "selected_selfies",
    "filtered_selfies",
    "used_filter_selfies_fallback",
    "decoded_smiles",
    "is_valid_selfies",
    "selfies_decode_error",
    "canonical_smiles",
    "derived_selfies",
    "is_valid_smiles",
    "best_reference_smiles",
    "max_dice_similarity",
    "passes_similarity_threshold",
    "rejection_reason",
]
print_records("Assessment preview", review_rows, limit=30, keys=detail_columns)

summary_rows = summarize_biot5_native_review_records(review_rows)
summary_rows = sorted(
    summary_rows,
    key=lambda item: (
        str(item["generation_config_name"]),
        str(item["description_id"]),
        str(item["prompt_variant"]),
    ),
)

summary_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "sample_count",
    "elapsed_seconds_batch",
    "seconds_per_sample_batch",
    "valid_selfies_rate",
    "filter_selfies_fallback_rate",
    "filter_selfies_recovery_rate",
    "valid_smiles_rate",
    "unique_canonical_smiles_count",
    "avg_max_dice_similarity",
    "best_max_dice_similarity",
    "passes_similarity_threshold_rate",
    "invalid_selfies_rate",
    "invalid_smiles_rate",
]
print_records("Summary rows", summary_rows, keys=summary_columns)

print_section("Compact Summary")
for summary in summary_rows:
    print(
        f"{summary['generation_config_name']} | {summary['description_id']} | {summary['prompt_variant']} | "
        f"samples={summary['sample_count']} | elapsed={summary['elapsed_seconds_batch']:.2f}s | "
        f"sec_per_sample={summary['seconds_per_sample_batch']:.3f} | "
        f"valid_selfies={summary['valid_selfies_rate']:.3f} | "
        f"filter_recovery={summary['filter_selfies_recovery_rate']:.3f} | "
        f"valid_smiles={summary['valid_smiles_rate']:.3f} | "
        f"unique={summary['unique_canonical_smiles_count']} | "
        f"avg_dice={summary['avg_max_dice_similarity']:.3f} | "
        f"best_dice={summary['best_max_dice_similarity']:.3f}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(RAW_GENERATIONS_PATH, raw_generations)
write_jsonl(REVIEW_ROWS_PATH, review_rows)
SUMMARY_ROWS_PATH.write_text(json_dumps(summary_rows), encoding="utf-8")
REVIEW_CONFIG_SNAPSHOT_PATH.write_text(json_dumps({
    "runtime": runtime_report,
    "hf_login_status": hf_login_status,
    "train_file": str(TRAIN_FILE),
    "model_name_or_path": MODEL_NAME_OR_PATH,
    "selected_description_count": SELECTED_DESCRIPTION_COUNT,
    "selection_seed": SELECTION_SEED,
    "selected_description_ids": SELECTED_DESCRIPTION_IDS,
    "num_samples": NUM_SAMPLES,
    "model_max_length": MODEL_MAX_LENGTH,
    "allow_filter_fallback": ALLOW_FILTER_FALLBACK,
    "generation_configs": GENERATION_CONFIGS,
}), encoding="utf-8")

review_artifact_dir, review_manifest = export_stage_artifacts(
    stage_name=REVIEW_STAGE_NAME,
    artifact_map={
        "mini_dataset": OUTPUT_DIR,
    },
    metadata={
        "runtime": runtime_report,
        "hf_login_status": hf_login_status,
        "model_name_or_path": MODEL_NAME_OR_PATH,
        "selected_description_count": SELECTED_DESCRIPTION_COUNT,
        "selection_seed": SELECTION_SEED,
        "selected_description_ids": SELECTED_DESCRIPTION_IDS,
        "num_samples": NUM_SAMPLES,
        "output_dir": str(OUTPUT_DIR),
    },
)
review_artifact_zip_path = create_zip_archive(review_artifact_dir, REVIEW_ARTIFACT_ZIP_PATH)

MINI_POST_TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MINI_POST_TRAINING_CONFIG_SNAPSHOT_PATH.write_text(json_dumps({
    "grouped_train_file": str(LOCAL_GROUPED_TRAIN_FILE),
    "copied_grouped_train": None if copied_grouped_train is None else str(copied_grouped_train),
    "mini_post_training_example_count": MINI_POST_TRAINING_EXAMPLE_COUNT,
    "selection_seed": SELECTION_SEED,
    "selected_grouped_example_ids": SELECTED_GROUPED_EXAMPLE_IDS,
    "validation_fraction": VALIDATION_FRACTION,
    "test_fraction": TEST_FRACTION,
    "mini_grouped_sample_path": str(MINI_GROUPED_SAMPLE_PATH),
    "mini_split_paths": {name: str(path) for name, path in mini_split_paths.items()},
    "mini_split_counts": {name: len(read_jsonl(path)) for name, path in mini_split_paths.items()},
}), encoding="utf-8")

mini_post_training_artifact_dir, mini_post_training_manifest = export_stage_artifacts(
    stage_name=MINI_POST_TRAINING_STAGE_NAME,
    artifact_map={
        "grouped_splits": MINI_GROUPED_SPLIT_DIR,
        "config_snapshot.json": MINI_POST_TRAINING_CONFIG_SNAPSHOT_PATH,
        "mini_grouped_train_multimol.jsonl": MINI_GROUPED_SAMPLE_PATH,
    },
    metadata={
        "grouped_train_file": str(LOCAL_GROUPED_TRAIN_FILE),
        "copied_grouped_train": None if copied_grouped_train is None else str(copied_grouped_train),
        "mini_post_training_example_count": MINI_POST_TRAINING_EXAMPLE_COUNT,
        "selection_seed": SELECTION_SEED,
        "selected_grouped_example_ids": SELECTED_GROUPED_EXAMPLE_IDS,
        "split_paths": {name: str(path) for name, path in mini_split_paths.items()},
    },
)
mini_post_training_zip_path = create_zip_archive(
    mini_post_training_artifact_dir,
    MINI_POST_TRAINING_ZIP_PATH,
)

print_json("Saved Outputs", {
    "raw_generations_path": str(RAW_GENERATIONS_PATH),
    "review_rows_path": str(REVIEW_ROWS_PATH),
    "summary_rows_path": str(SUMMARY_ROWS_PATH),
    "review_config_snapshot_path": str(REVIEW_CONFIG_SNAPSHOT_PATH),
    "review_artifact_dir": str(review_artifact_dir),
    "review_artifact_zip_path": str(review_artifact_zip_path),
    "review_manifest": review_manifest,
    "mini_grouped_sample_path": str(MINI_GROUPED_SAMPLE_PATH),
    "mini_post_training_config_snapshot_path": str(MINI_POST_TRAINING_CONFIG_SNAPSHOT_PATH),
    "mini_split_paths": {name: str(path) for name, path in mini_split_paths.items()},
    "mini_post_training_artifact_dir": str(mini_post_training_artifact_dir),
    "mini_post_training_zip_path": str(mini_post_training_zip_path),
    "mini_post_training_manifest": mini_post_training_manifest,
})


## Review Questions

After the notebook runs, use the summary, saved review outputs, and zipped artifact bundle to answer:

1. Does the native BioT5 path stay stable across a 128-description random sample instead of only a few hand-picked prompts?
2. How often is `filter_selfies(...)` still needed to recover otherwise valid outputs in the mini-dataset setting?
3. Does diverse beam improve usable molecule diversity without collapsing validity on this broader random slice?
4. Is the exported `mini_dataset` artifact bundle enough for offline local analysis without a merge stage?
